In [18]:
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings,ChatHuggingFace,HuggingFaceEndpoint
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor

In [13]:
docs = [
    Document(
        page_content="""
        The Gran canyon is the one of the most visited natural wonders in the world .
        Photosynthesis is the process by which green plants convert sunlight into chemical energy.
        Millions of people travel to see it every year. It is located in Arizona, USA.
        """
    ),
    Document(
        page_content="""
        MS Dhoni is a former Indian captain.
        He plays for Chennai Super Kings.
        Dhoni is known for calm leadership.
        """
    ),
    Document(
        page_content="""
        Basketball was invented by Dr. James Naismith in 1891.
        It was originally played with a soccer ball and two peach baskets.NBA is now the premier basketball league in the world.
        """
    ),
    Document(
        page_content="""
        The history of Cinema began in the late 1800s. Silent film era was from 1890s to late 1920s. The Golden Age of Hollywood was from the late 1920s to early 1960s.
        photosynthesis is the process by which green plants and some other organisms use sunlight to synthesize foods with the help of chlorophyll. It is a crucial process for life on Earth.
        modern filmaking involves complex CGI techniques.
        """
    )
]

In [14]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6989.60it/s]


In [16]:
vector_store = FAISS.from_documents(
    docs,
    embeddings
)

In [20]:
base_retriever = vector_store.as_retriever(search_kwargs={"k": 2})

In [19]:
# set up compression using LLM 
llm = ChatHuggingFace(llm=HuggingFaceEndpoint(
        repo_id="meta-llama/Llama-3.1-8B-Instruct",
        task="text-generation",
        max_new_tokens=512,
        temperature=0.1
    ))
compressor = LLMChainExtractor.from_llm(llm)

In [22]:
# create contextual compression retriever
compressed_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=base_retriever
)

In [23]:
query  = "what is photosynthesis ?"
compressed_result = compressed_retriever.invoke(query)

In [24]:
for i , doc in enumerate(compressed_result):
    print(f"Compressed Document {i+1}:\n{doc.page_content}\n")

Compressed Document 1:
photosynthesis is the process by which green plants and some other organisms use sunlight to synthesize foods with the help of chlorophyll. It is a crucial process for life on Earth.

Compressed Document 2:
Photosynthesis is the process by which green plants convert sunlight into chemical energy.

